# Fan-Sim Colab - No GPU


In [ ]:
# Mount Google Drive for persistent input, checkpoints, and output sync.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import shutil
import subprocess

REPO_URL = 'https://github.com/tcthuong/fan-sim.git'
PROJECT_ROOT = Path('/content/fan-sim')
DRIVE_ROOT = Path('/content/drive/MyDrive/colab-data/fan-sim')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
SYNC_ROOT = DRIVE_ROOT / 'outputs'

if not Path('/content/drive/MyDrive').exists():
    raise RuntimeError('Mount Google Drive before configuring checkpoints and sync.')

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SYNC_ROOT.mkdir(parents=True, exist_ok=True)


def _now_iso():
    return datetime.now(timezone.utc).isoformat()


def _has_data(path):
    path = Path(path)
    if not path.exists():
        return False
    if path.is_file():
        return path.stat().st_size > 0
    return any(path.iterdir())


def _project_path(path):
    path = Path(path)
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path


def _sync_destination(src):
    src = Path(src)
    try:
        rel = src.resolve().relative_to(PROJECT_ROOT.resolve())
    except ValueError:
        rel = Path(src.name)
    return SYNC_ROOT / rel


def _copy_tree_or_file(src, dst, delete=True):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_file():
        shutil.copy2(src, dst)
    elif shutil.which('rsync'):
        dst.mkdir(parents=True, exist_ok=True)
        args = ['rsync', '-a']
        if delete:
            args.append('--delete')
        args.extend([f'{src}/', f'{dst}/'])
        subprocess.run(args, check=True)
    else:
        if delete and dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst, dirs_exist_ok=not delete)


def sync_to_drive(*paths):
    synced = []
    for raw_path in paths:
        src = _project_path(raw_path)
        if not _has_data(src):
            print(f'[sync] skip empty/missing: {src}')
            continue

        dst = _sync_destination(src)
        _copy_tree_or_file(src, dst, delete=True)
        synced.append(str(dst))
        print(f'[sync] {src} -> {dst}')
    return synced


def sync_from_drive(*paths):
    restored = []
    for raw_path in paths:
        dst = _project_path(raw_path)
        src = _sync_destination(dst)
        if not _has_data(src):
            raise FileNotFoundError(f'No synced data in Google Drive: {src}')

        _copy_tree_or_file(src, dst, delete=True)
        restored.append(str(dst))
        print(f'[restore] {src} -> {dst}')
    return restored


def mark_checkpoint(step, **metadata):
    checkpoint = CHECKPOINT_DIR / f'{step}.json'
    payload = {
        'step': step,
        'completed_at': _now_iso(),
        'project_root': str(PROJECT_ROOT),
        'sync_root': str(SYNC_ROOT),
        'metadata': metadata,
    }
    tmp = checkpoint.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
    tmp.replace(checkpoint)
    print(f'[checkpoint] {checkpoint}')
    return checkpoint


def checkpoint_and_sync(step, *paths, **metadata):
    checkpoint = mark_checkpoint(step, **metadata)
    synced = sync_to_drive(*paths)
    return {'checkpoint': str(checkpoint), 'synced': synced}


def run(command, cwd=PROJECT_ROOT):
    print(f'$ {command}')
    subprocess.run(command, shell=True, check=True, cwd=str(cwd), executable='/bin/bash')


def run_step(step, commands, sync_paths=(), **metadata):
    for command in commands:
        run(command)
    return checkpoint_and_sync(step, *sync_paths, **metadata)

print(f'Checkpoint dir: {CHECKPOINT_DIR}')
print(f'Sync dir: {SYNC_ROOT}')

## No GPU workflow

Run this notebook on a CPU runtime to produce data and sync outputs to Google Drive. The GPU notebook restores the synced graph data instead of recreating it.

Clone or pull the project from GitHub. Keep OpenFOAM input data and generated artifacts outside git; copy the base case into data/base_case after the repo is ready.

### Pull GitHub repo

Run this step in every fresh Colab runtime. It clones the repo if missing, otherwise pulls the latest code with `git pull --ff-only`.

In [ ]:
# Pull GitHub repo: clone on a new runtime, pull when the repo already exists.
if (PROJECT_ROOT / '.git').is_dir():
    run(f'git -C "{PROJECT_ROOT}" pull --ff-only', cwd=Path('/content'))
else:
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    run(f'git clone "{REPO_URL}" "{PROJECT_ROOT}"', cwd=Path('/content'))

os.chdir(PROJECT_ROOT)
print(Path.cwd())
run('python --version')
checkpoint_and_sync('01_repo_ready')

In [ ]:
src = Path('/content/drive/MyDrive/colab-data/data/base_case')
dst = PROJECT_ROOT / 'data/base_case'

if not src.exists():
    raise FileNotFoundError(f'Missing source folder: {src}')

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print('Copied base_case from:')
print(src)
print('to:')
print(dst)
checkpoint_and_sync('02_base_case_ready', 'data/base_case')

In [ ]:
# Smoke install. Use this first because it avoids the large PhysicsNeMo/Torch install.
run_step(
    '03_smoke_install',
    [
        'python -m pip install --upgrade pip',
        'python -m pip install -e ".[service,vtk,dev]"',
        'python -m pytest -q',
        'fan-sim --help',
    ],
)

In [ ]:
# Install OpenFOAM 2406. This can take several minutes and can fail if Colab changes its Ubuntu image.
run_step('04_openfoam_install', ['bash scripts/colab_install_openfoam2406.sh'])

In [ ]:
# Generate the one-case Colab smoke matrix.
run_step(
    '05_generate_cases',
    ['fan-sim generate-cases --config configs/fan_sim_colab.yaml'],
    sync_paths=['runs/openfoam/case_rpm_0600_pout_000'],
)

In [ ]:
# Patch generated controlDict to a two-iteration smoke run.
p = Path('runs/openfoam/case_rpm_0600_pout_000/system/controlDict')
s = p.read_text()
s = s.replace('endTime 1000.0;', 'endTime 2;')
s = s.replace('writeInterval 1000;', 'writeInterval 1;')
p.write_text(s)
print(p)
checkpoint_and_sync('06_patch_smoke_control_dict', 'runs/openfoam/case_rpm_0600_pout_000')

In [ ]:
run_step(
    '07_run_openfoam',
    [
        'fan-sim run-openfoam --config configs/fan_sim_colab.yaml --case-id case_rpm_0600_pout_000',
        'tail -120 runs/openfoam/case_rpm_0600_pout_000/log.fan-sim-openfoam',
    ],
    sync_paths=['runs/openfoam/case_rpm_0600_pout_000'],
)

In [ ]:
run_step(
    '08_export_vtk',
    [
        'fan-sim export-vtk --config configs/fan_sim_colab.yaml --case-id case_rpm_0600_pout_000',
        'find runs/openfoam/case_rpm_0600_pout_000/VTK -maxdepth 3 -name "*.vtu" -print',
    ],
    sync_paths=['runs/openfoam/case_rpm_0600_pout_000/VTK'],
)

In [ ]:
run_step(
    '09_build_graphs',
    [
        'fan-sim build-graphs --config configs/fan_sim_colab.yaml',
        'find artifacts/graphs -name "*.graph.pt" -print',
    ],
    sync_paths=['artifacts/graphs'],
)